# Multi-Modal Vision Transformer (ViT) Demo Notebook
This notebook demonstrates end-to-end usage of the Multi-Modal Vision Transformer for:
1. **Image Captioning** (Greedy & Beam Search)
2. **Visual Question Answering (VQA)**
3. **Cross-Modal Image-Text Retrieval**

In [ ]:
import torch
from PIL import Image
import numpy as np
from models.multimodal_model import MultiModalViT
from inference.caption import ImageCaptioner
from inference.vqa import VQAEngine
from inference.retrieval import CrossModalRetriever
from utils.tokenizer import MultiModalTokenizer

# 1. Initialize Tokenizer & Model
tokenizer = MultiModalTokenizer(model_name="bert-base-uncased")
model = MultiModalViT(hidden_dim=256, projection_dim=128, num_fusion_layers=2, num_heads=4)
model.eval()

print("✅ MultiModalViT model successfully initialized!")

In [ ]:
# 2. Create Synthetic Test Image
img_array = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
sample_image = Image.fromarray(img_array)

# 3. Run Image Captioning
captioner = ImageCaptioner(model=model, tokenizer=tokenizer)
caption = captioner.generate_caption(sample_image, decoding_strategy="beam_search", beam_size=3)
print(f"🖼️ Generated Caption: '{caption}'")

In [ ]:
# 4. Run Visual Question Answering (VQA)
vqa_engine = VQAEngine(model=model, tokenizer=tokenizer, answer_vocab=["red", "blue", "yes", "dog"])
vqa_res = vqa_engine.predict_answer(sample_image, question="What color is the object?", top_k=3)
print(f"❓ Question: 'What color is the object?'")
print(f"💡 Answer: '{vqa_res['answer']}' (Confidence: {vqa_res['confidence']*100:.1f}%)")

In [ ]:
# 5. Run Cross-Modal Retrieval
retriever = CrossModalRetriever(model=model, tokenizer=tokenizer)
retriever.index_dataset([sample_image], ["A red car on a highway.", "A dog in a park."])
search_res = retriever.search_texts_by_image(sample_image, top_k=2)
print("🔎 Retrieval Results:", search_res)